# Runnables — Databricks Native

In [ ]:
%run databricks_native_common.py

## 1. RunnablePassthrough — passes input unchanged

In [ ]:
pt = RunnablePassthrough()
print(pt.invoke("Hello"))  # -> "Hello"

## 2. Custom Runnable via RunnableLambda

In [ ]:
add_one = RunnableLambda(lambda x: x + 1)
double  = RunnableLambda(lambda x: x * 2)

# Chain them: (5 + 1) * 2 = 12
pipeline = add_one | double
print(pipeline.invoke(5))  # -> 12

## 3. RunnablePassthrough.assign() — enriching a dict

In [ ]:
def extract_city(question: str) -> str:
    """Toy extractor — real use would call an LLM or NER model."""
    for city in ["Seattle", "New York", "London", "Tokyo"]:
        if city.lower() in question.lower():
            return city
    return "Unknown"

pre = (
    RunnableLambda(lambda x: {"question": x})
    | RunnablePassthrough.assign(city=lambda d: extract_city(d["question"]))
)

result = pre.invoke("What is the weather like in Seattle today?")
print(result)  # {'question': '...', 'city': 'Seattle'}

## 4. Enrichment into an LLM prompt

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a weather assistant."),
    ("human", "The user asked: {question}\nDetected city: {city}\nProvide a short mock weather report."),
])

chain = pre | prompt | llm_noreason | StrOutputParser()
print(chain.invoke("What is the weather like in Tokyo?"))

## 5. Multi-field enrichment

In [ ]:
enrich_chain = (
    RunnableLambda(lambda text: {"text": text})
    | RunnablePassthrough.assign(
        char_count =lambda d: len(d["text"]),
        word_count =lambda d: len(d["text"].split()),
        is_question=lambda d: d["text"].strip().endswith("?"),
        uppercase  =lambda d: d["text"].upper(),
    )
)

from pprintpp import pprint
pprint(enrich_chain.invoke("What is Databricks Vector Search?"))

## 6. RunnableSequence (explicit form of `|`)

In [ ]:
simple_prompt = ChatPromptTemplate.from_messages([
    ("human", "Summarise {topic} in one sentence."),
])

# These two are equivalent:
chain_pipe  = simple_prompt | llm | StrOutputParser()
chain_seq   = RunnableSequence(simple_prompt, llm, StrOutputParser())

print(chain_pipe.invoke({"topic": "Databricks Vector Search"}))